In [64]:
import mdtraj as md

from sys import stdout

# OpenMM imports
import openmm.app as app
import openmm as mm
import openmm.unit as unit
from openmmforcefields.generators import SMIRNOFFTemplateGenerator

# OpenFF-toolkit imports
from openff.toolkit import Molecule
from openff.toolkit import Topology as offTopology
from openff.units.openmm import to_openmm as offquantity_to_openmm

import parmed as pmd

import subprocess

In [ ]:
def PBSA_calc(traj_file, top_file, cond):
    traj = md.load(traj_file, top=top_file)

    # remove all other atoms except protein
    protein_atoms = traj.topology.select('protein and not resname UNK')

    traj_protein = traj.atom_slice(protein_atoms)

    traj_protein.save('traj_protein.dcd')
    traj_protein[0].save('traj_protein.pdb')

    protein_pdb = app.PDBFile('traj_protein.pdb')
    ff = app.ForceField('amber/protein.ff14SB.xml', 'amber/tip3p_standard.xml')
    modeller = app.Modeller(protein_pdb.topology, protein_pdb.positions)
    system = ff.createSystem(modeller.topology)

    structure = pmd.openmm.load_topology(protein_pdb.topology, system=system, xyz=protein_pdb.positions)

    structure.save('complex.prmtop', overwrite=True)
    #structure.save('complex.inpcrd', overwrite=True)
    #structure.save('traj.rst7', overwrite=True)

    rec_atoms = traj.topology.select('chainid 0')  # chain A
    rec_traj = traj_protein.atom_slice(rec_atoms)
    rec_traj[0].save('receptor.pdb')
    rec_pdb = app.PDBFile('receptor.pdb')
    modeller = app.Modeller(rec_pdb.topology, rec_pdb.positions)
    system = ff.createSystem(modeller.topology)
    rec_struct = pmd.openmm.load_topology(rec_pdb.topology, system=system, xyz=rec_pdb.positions)
    rec_struct.save('receptor.prmtop', overwrite=True)

    lig_atoms = traj.topology.select('chainid 1')  # chain B
    lig_traj = traj_protein.atom_slice(lig_atoms)
    lig_traj[0].save('ligand.pdb')
    lig_pdb = app.PDBFile('ligand.pdb')
    modeller = app.Modeller(lig_pdb.topology, lig_pdb.positions)
    system = ff.createSystem(modeller.topology)
    lig_struct = pmd.openmm.load_topology(lig_pdb.topology, system=system, xyz=lig_pdb.positions)
    lig_struct.save('ligand.prmtop', overwrite=True)

    infile = "mmpbsa.in"

    mmpbsa_content = """&general
    startframe=51,
    endframe=250,
    interval=5,
    verbose=1,
    /
    &decomp
    idecomp=1,
    dec_verbose=1,
    /
    &pb
    istrng=0.15,
    /
    """

    with open(infile, "w+") as f:
        f.write(mmpbsa_content)

    cmd = [
        "MMPBSA.py",
        "-O",
        "-i", "mmpbsa.in",
        "-cp", "complex.prmtop",
        "-rp", "receptor.prmtop",
        "-lp", "ligand.prmtop",
        "-y", "traj_protein.dcd",
        "-o", f"output/o_{cond}.dat",
        "-eo", f"output/eo_{cond}.dat",
        "-do", f"output/do_{cond}.dat",
        "-deo", f"output/deo_{cond}.dat",
    ]

    subprocess.run(cmd, capture_output=True, text=True)

In [66]:
sample_list = [['traj_atp_processed.dcd','topology_atp.pdb','atp'],['traj_fexofenadine_processed.dcd','topology_fexofenadine.pdb','fexofenadine'],['traj_osu-t315_processed.dcd','topology_osu-t315.pdb','osu-t315']]

for sample in sample_list:
    PBSA_calc('input/' + sample[0], 'input/' + sample[1], sample[2])

dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
dcdplugin) detected standard 32-bit DCD file of native endianness
dcdplugin) CHARMM format DCD file (also NAMD 2.1 and later)
